# APEX quickstart

**APEX co-designs PolyGIN and exact quadrature to produce architecture-certified Aumann-Shapley explanations for graphs.**

This notebook runs a complete explanation workflow on one tiny graph: prepare a graph, supply a compatible polynomial predictor, explain its predicted class, inspect signed node scores, and visualize the result. It downloads no data and trains no model.

## 1. Imports and reproducibility

Install the repository in editable mode before running the notebook: `python -m pip install --no-deps --no-build-isolation -e .`

In [ ]:
import torch
from torch_geometric.data import Data

from apex import APEX

torch.manual_seed(7)
device = torch.device("cpu")
print("device:", device)

## 2. Prepare one graph

APEX accepts any single graph object exposing `x` (`[num_nodes, num_features]`) and `edge_index` (`[2, num_edges]`). The edges below are stored in both directions.

In [ ]:
x = torch.tensor([
    [ 1.0,  0.2, -0.5],
    [ 0.4, -0.1,  0.3],
    [-0.7,  0.6,  0.2],
    [ 0.2, -0.4,  0.8],
    [ 0.9,  0.1, -0.2],
], dtype=torch.float32)

edge_index = torch.tensor([
    [0, 1, 1, 2, 2, 3, 3, 4, 0, 4],
    [1, 0, 2, 1, 3, 2, 4, 3, 4, 0],
], dtype=torch.long)

graph = Data(x=x, edge_index=edge_index).to(device)
print(graph)

## 3. Supply a compatible polynomial predictor

To keep this quickstart deterministic and independent of datasets or checkpoints, the demo predictor below has two quadratic class logits. It is a transparent stand-in for a trained `PolyGIN`; the explanation path and public APEX API are the same. Because its maximum polynomial depth is two, we will use `depth=2`.

In [ ]:
class DemoPolynomialModel(torch.nn.Module):
    def forward(self, x, edge_index, batch=None):
        class_0 = (x[:, 0].square() + 2.0 * x[:, 1]).sum()
        class_1 = (-0.5 * x[:, 0].square() + x[:, 2]).sum()
        return torch.stack((class_0, class_1)).unsqueeze(0)

model = DemoPolynomialModel().to(device).eval()
with torch.no_grad():
    logits = model(graph.x, graph.edge_index)
print("logits:", logits)
print("predicted class:", int(logits.argmax(dim=-1).item()))

## 4. Explain the prediction

`target=None` means: explain the class predicted by the model. `sparsity=0.4` keeps the top 60% of nodes in the binary explanation mask. Signed `node_scores` retain direction; positive values support the target logit relative to the zero baseline, while negative values oppose it.

In [ ]:
explainer = APEX(model, depth=2, device=device)
explanation = explainer.explain(graph, target=None, sparsity=0.4)

print("predicted class:    ", explanation.predicted_class)
print("explained class:    ", explanation.target_class)
print("quadrature points:  ", explanation.quadrature_points)
print("node scores:        ", explanation.node_scores.tolist())
print("selected-node mask: ", explanation.node_mask.tolist())
print("top-3 nodes:        ", explanation.top_nodes(3).tolist())
print("completeness gap:   ", explanation.completeness_gap)

## 5. Visualize signed node attribution

The optional plot uses blue for positive support and red for negative evidence. If Matplotlib is not installed, the numerical result above is still complete.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    print("Install matplotlib to display the optional attribution chart.")
else:
    scores = explanation.node_scores.numpy()
    colors = ["#2563eb" if value >= 0 else "#dc2626" for value in scores]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar(range(len(scores)), scores, color=colors)
    ax.axhline(0, color="#111827", linewidth=0.8)
    ax.set(xlabel="Node index", ylabel="Signed attribution", title="APEX node explanation")
    ax.set_xticks(range(len(scores)))
    plt.show()

## 6. Replace the demo model with a trained PolyGIN

For a real experiment, keep the explanation cell unchanged and replace only `graph` and `model`:

```python
from apex.models.gnn import PolyGIN
from apex.utils.checkpoints import compatible_state_dict

model = PolyGIN(
    model_level="graph",
    dim_node=graph.x.size(1),
    dim_hidden=300,
    num_classes=2,
).to(device)
state = torch.load("artifacts/checkpoints/<dataset>/PolyGIN_seed0.pkl", map_location=device, weights_only=True)
model.load_state_dict(compatible_state_dict(state))
model.eval()

explanation = APEX(model, depth=4, device=device).explain(graph)
```

The `depth` value must match the polynomial transformation depth of the model. Exactness is guaranteed only for compatible polynomial scores under the assumptions stated in the paper.